In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Generalizability Evaluation

## Overview
This notebook evaluates whether the findings in the repository generalize beyond the original experimental setting.

## Checklist Items:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data
- **GT3**: Method / Specificity Generalizability

## Repository: `/net/scratch2/smallyan/rome_eval`

In [2]:
# First, let's explore the repository structure
repo_path = '/net/scratch2/smallyan/rome_eval'

# List the top-level structure
for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:  # Only go 2 levels deep for initial exploration
        subindent = ' ' * 2 * (level + 1)
        for file in files[:10]:  # Limit files shown
            print(f'{subindent}{file}')
        if len(files) > 10:
            print(f'{subindent}... and {len(files) - 10} more files')

rome_eval/
  globals.yml
  CodeWalkthrough.md
  .gitignore
  plan.md
  CITATION.cff
  documentation.pdf
  LICENSE
  .gitattributes
  util/
    globals.py
    __init__.py
    hparams.py
    runningstats.py
    nethook.py
    generate.py
    perplexity.py
    logit_lens.py
    __pycache__/
  hparams/
    FT/
    KE/
    MEND/
    ROME/
    KN/
  rome/
    rome_main.py
    tok_dataset.py
    __init__.py
    repr_tools.py
    README.md
    rome_hparams.py
    compute_u.py
    compute_v.py
    layer_stats.py
    __pycache__/
  .git/
    COMMIT_EDITMSG
    index
    config
    ORIG_HEAD
    FETCH_HEAD
    description
    packed-refs
    HEAD
    refs/
      heads/
      remotes/
        origin/
      tags/
    info/
    logs/
      refs/
        heads/
        remotes/
          origin/
    objects/
      b5/
      40/
      cf/
      02/
      75/
      f7/
      2c/
      e5/
      11/
      88/
      pack/
      7e/
      54/
      23/
      d6/


      27/
      86/
      68/
      1e/
      95/
      a4/
      de/
      76/
      1b/
      ef/
      18/
      0a/
      c8/
      ac/
      10/
      bc/
      ed/
      d7/
      0c/
      5e/
      b3/
      8d/
      41/
      79/
      5b/
      a6/
      73/
      info/
      c7/
      52/
      fa/
      ec/
      3f/
      21/
      9c/
      d4/
      77/
      a2/
      cc/
      5f/
      93/
      b0/
      1c/
      07/
      a5/
      cd/
      dc/
      f8/
      d3/
      63/
    hooks/
  dsets/
    __init__.py
    tfidf_stats.py
    zsre.py
    attr_snippets.py
    counterfact.py
    knowns.py
    __pycache__/
  experiments/
    evaluate.py
    summarize.py
    __init__.py
    causal_trace.py
    sweep.py


    py/
      __pycache__/
    __pycache__/
  evaluation/
    consistency_evaluation.json
    self_matching.ipynb
    test_heatmap.pdf
    replications/
      2026-01-11_12-20-56/
  notebooks/
    globals.yml
    average_causal_effects.ipynb
    causal_trace.ipynb
    causal_trace_frozen_mlp_attn.ipynb
    rome.ipynb
    vis/
  baselines/
    README.md
    ft/
      __pycache__/
    efk/
      __pycache__/
    kn/
      __pycache__/
      knowledge_neurons/
        tests/
        knowledge_neurons/
          __pycache__/
    mend/
      config/
        model/
        alg/
        experiment/
      __pycache__/
      algs/
        __pycache__/
      data_classes/


  data/
    attribute_snippets.json
    idf.npy
    tfidf_vocab.json
    counterfact.json
    known_1000.json
    stats/
      gpt2/
        wikipedia_stats/
      gpt2-xl/
        wikipedia_stats/
  scripts/
    setup_clean_ipynb.sh
    setup_conda.sh
    rome.yml
    causal_trace.sh
    ipynb_drop_output.py
    collect_layer_stats.sh
    colab_reqs/


In [3]:
# Let's read the plan.md and documentation to understand the research
with open(os.path.join(repo_path, 'plan.md'), 'r') as f:
    plan_content = f.read()
print("=== PLAN.MD ===")
print(plan_content[:5000])

=== PLAN.MD ===
# Plan
## Objective
Analyze the storage and recall of factual associations in autoregressive transformer language models, finding evidence that these associations correspond to localized, directly-editable computations.

## Hypothesis
1. Factual associations in GPT correspond to a localized computation mechanism where each midlayer MLP module accepts inputs encoding a subject, then produces outputs recalling memorized properties about that subject, with middle layer MLP outputs accumulating information that is copied to the last token by attention at high layers.
2. Factual associations are localized in the MLP modules at specific middle layers, specifically at the processing of the subject's last token.
3. MLP layers in transformers can be modeled as linear associative memory where weights act as key-value stores.

## Methodology
1. Develop a causal intervention method (Causal Tracing) using causal mediation analysis to identify neuron activations that are decisive in 

In [4]:
# Read more of the plan
print(plan_content[5000:])

In [5]:
# Let's check the CodeWalkthrough to understand the implementation
with open(os.path.join(repo_path, 'CodeWalkthrough.md'), 'r') as f:
    walkthrough = f.read()
print("=== CODE WALKTHROUGH ===")
print(walkthrough[:8000])

=== CODE WALKTHROUGH ===
# Rank-One Model Editing (ROME)

This repository provides an implementation of Rank-One Model Editing (ROME) on auto-regressive transformers (GPU-only).
We currently support OpenAI's GPT-2 XL (1.5B) and EleutherAI's GPT-J (6B). The release of a 20B GPT-like model from EleutherAI is expected soon; we hope to support it ASAP.

Feel free to open an issue if you find any problems; we are actively developing this repository and will monitor tickets closely.

[![Colab ROME Demo](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb)

<p align="center">
    <img src="https://rome.baulab.info/images/eiftower-crop.svg" alt="causal tracing GIF" width="425px" />
</p>

## Table of Contents
1. [Installation](#installation)
2. [Causal Tracing](#causal-tracing)
3. [Rank-One Model Editing (ROME)](#rank-one-model-editing-rome-1)
4. [CounterFact](#counterfact)
5. [Evaluation](#evaluation)
 

## Understanding the Research

From the plan.md and CodeWalkthrough.md, the research focuses on:

### Key Findings:
1. **Neuron-level finding**: Factual associations in GPT models are localized in MLP modules at specific middle layers (around layer 15-18) at the last subject token
2. **Mechanism**: MLP modules accept inputs encoding a subject and produce outputs recalling memorized properties about that subject
3. **Method proposed**: 
   - **Causal Tracing**: Identifies neuron activations decisive in factual predictions
   - **ROME (Rank-One Model Editing)**: Modifies feed-forward weights to update specific factual associations

### Original Models Used:
- GPT-2 XL (1.5B parameters)
- GPT-J (6B parameters)

### Original Datasets:
- zsRE (zero-shot relation extraction)
- COUNTERFACT dataset

In [6]:
# Let's check the original dataset used
import json

# Load the counterfact dataset
counterfact_path = os.path.join(repo_path, 'data', 'counterfact.json')
with open(counterfact_path, 'r') as f:
    counterfact = json.load(f)

print(f"Number of COUNTERFACT records: {len(counterfact)}")
print("\nExample record:")
print(json.dumps(counterfact[0], indent=2))

Number of COUNTERFACT records: 21919

Example record:
{
  "case_id": 0,
  "pararel_idx": 2796,
  "requested_rewrite": {
    "prompt": "The mother tongue of {} is",
    "relation_id": "P103",
    "target_new": {
      "str": "English",
      "id": "Q1860"
    },
    "target_true": {
      "str": "French",
      "id": "Q150"
    },
    "subject": "Danielle Darrieux"
  },
  "paraphrase_prompts": [
    "Shayna does this and Yossel goes still and dies. Danielle Darrieux, a native",
    "An album was recorded for Capitol Nashville but never released. Danielle Darrieux spoke the language"
  ],
  "neighborhood_prompts": [
    "The mother tongue of L\u00e9on Blum is",
    "The native language of Montesquieu is",
    "Fran\u00e7ois Bayrou, a native",
    "The native language of Raymond Barre is",
    "Michel Rocard is a native speaker of",
    "Jacques Chaban-Delmas is a native speaker of",
    "The native language of Fran\u00e7ois Bayrou is",
    "Maurice Genevoix, speaker of",
    "The mother 

In [7]:
# Look at the causal tracing implementation to understand the method
causal_trace_path = os.path.join(repo_path, 'experiments', 'causal_trace.py')
with open(causal_trace_path, 'r') as f:
    causal_trace_code = f.read()
print(causal_trace_code[:5000])

import argparse
import json
import os
import re
from collections import defaultdict

import numpy
import torch
from datasets import load_dataset
from matplotlib import pyplot as plt
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from dsets import KnownsDataset
from rome.tok_dataset import (
    TokenizedDataset,
    dict_to_,
    flatten_masked_batch,
    length_collation,
)
from util import nethook
from util.globals import DATA_DIR
from util.runningstats import Covariance, tally


def main():
    parser = argparse.ArgumentParser(description="Causal Tracing")

    def aa(*args, **kwargs):
        parser.add_argument(*args, **kwargs)

    def parse_noise_rule(code):
        if code in ["m", "s"]:
            return code
        elif re.match("^[uts][\d\.]+", code):
            return code
        else:
            return float(code)

    aa(
        "--model_name",
        default="gpt2-xl",
        choices=[
            "gpt2-xl",
            "Eleut

In [8]:
# Let's check the known_1000.json file which is the original dataset
known_path = os.path.join(repo_path, 'data', 'known_1000.json')
with open(known_path, 'r') as f:
    knowns = json.load(f)

print(f"Number of known facts: {len(knowns)}")
print("\nExample records:")
for i in range(3):
    print(json.dumps(knowns[i], indent=2))

Number of known facts: 1209

Example records:
{
  "known_id": 0,
  "subject": "Vinson Massif",
  "attribute": "Antarctica",
  "template": "{} is located in the continent",
  "prediction": " of Antarctica. It is the largest of the three",
  "prompt": "Vinson Massif is located in the continent of",
  "relation_id": "P30"
}
{
  "known_id": 1,
  "subject": "Beats Music",
  "attribute": "Apple",
  "template": "{} is owned by",
  "prediction": " Apple, which is also the owner of Beats Electronics",
  "prompt": "Beats Music is owned by",
  "relation_id": "P127"
}
{
  "known_id": 2,
  "subject": "Audible.com",
  "attribute": "Amazon",
  "template": "{} is owned by",
  "prediction": " Amazon.com, Inc. or its affiliates.",
  "prompt": "Audible.com is owned by",
  "relation_id": "P127"
}


In [9]:
# Now let's set up the environment and check available resources
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe
CUDA memory: 85.09 GB


In [10]:
# Let's load environment variables for API access
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if 'TOKEN' in line or 'API' in line or 'HF' in line:
        key = line.split('=')[0]
        print(f"Found: {key}")

Found: OPENAI_API_KEY
Found: VSCODE_GIT_IPC_AUTH_TOKEN
Found: HF_TOKEN
Found: HF_HOME
Found: NDIF_API_KEY
Found: OPENROUTER_API_KEY


In [11]:
# Load the HF token
import subprocess

def get_env_var(var_name):
    result = subprocess.run(['bash', '-c', f'source /home/smallyan/.bashrc && echo ${var_name}'], 
                          capture_output=True, text=True)
    return result.stdout.strip()

hf_token = get_env_var('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print(f"HF Token loaded: {'Yes' if hf_token else 'No'}")

HF Token loaded: Yes
